# Pipeline de preprocessing — Adult Income (UCI)

**Objectif :** ce notebook ne fait pas d'analyse (elle est dans l'EDA) — il **vérifie**
que la pipeline (`adult_income.preprocessing.PreprocessingPipeline`) **applique fidèlement
les décisions du bilan AED** (un traitement *par feature*) et qu'elle **généralise au test
sans incohérence** (pas de fuite). À chaque décision du bilan correspond ici une vérification.

La pipeline est un `ColumnTransformer` scikit-learn composé de 6 branches :

| Branche | Colonnes | Traitement |
|---|---|---|
| `num` | numériques « normales » (age, education-num, hours-per-week) | médiane + `StandardScaler` |
| `num_log` | `capital-gain` | médiane + `log1p` + `RobustScaler` (queue lourde conservée) |
| `num_bin` | `capital-gain`, `capital-loss` | indicateur binaire `has_*` (décomposition « hurdle ») |
| `cat` | catégorielles faible cardinalité (marital-status, relationship, race, sex) | mode + `OneHotEncoder` |
| `cat_unknown` | `workclass`, `occupation` | catégorie **`Unknown`** + `OneHotEncoder` |
| `cat_country` | `native-country` | mode + regroupement des modalités `<50` + `OneHotEncoder` |

Points clés du bilan AED :
- **`capital-loss`** n'est plus log-transformée : un simple indicateur binaire suffit (magnitude peu informative). **`capital-gain`** garde le `log1p` **et** reçoit un indicateur binaire.
- **`workclass` / `occupation`** : la manquance est **informative** (~10 % `>50K` contre 24 % en moyenne), on la conserve via une catégorie `Unknown` au lieu de l'imputer par le mode.
- **`native-country`** : manquance non informative → imputation par le mode (US) ; ses ~20 modalités rares (`<50` obs) sont regroupées.
- **`education` / `fnlwgt`** : supprimées (`remainder="drop"`).

> ⚠️ **Hurdle vs naïf.** Ce notebook illustre la pipeline **par défaut**, c.-à-d. le
> traitement **`hurdle`** de `capital-*` (= la *proposition* AED). L'**ablation**
> (`modeling.ipynb §3`) montre que le hurdle **dégrade les modèles à arbres** → le pipeline
> **final retient le traitement `naive`** (`capital-*` = simples numériques mises à l'échelle).
> On garde ici le hurdle pour **visualiser la proposition** (notamment le `log1p`).

> ⚠️ La pipeline est **ajustée sur le train uniquement** (médianes, modes, échelles),
> puis appliquée au test. L'ajuster sur le test ferait fuiter de l'information du test
> dans le prétraitement (*data leakage*).

## 0. Préparation de l'environnement

In [ ]:
# Recharge automatique des modules src/ modifiés sans restart du kernel
%load_ext autoreload
%autoreload 2

# Permet d'importer le package src/ depuis le dossier notebooks/
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from adult_income import (
    DataLoader,
    PreprocessingPipeline,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET,
    POSITIVE_CLASS,
    NEGATIVE_CLASS,
)
from adult_income.data.schemas import (
    NORMAL_NUMERICAL_FEATURES,
    LOG_NUMERICAL_FEATURES,
    BINARY_NUMERICAL_FEATURES,
    MODE_CATEGORICAL_FEATURES,
    UNKNOWN_ONEHOT_FEATURES,
    GROUPED_CATEGORICAL_FEATURES,
    RARE_CATEGORY_MIN_FREQUENCY,
    FEATURES_TO_DROP,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Imports OK")


## 1. Chargement et découpage train / test

In [ ]:
loader = DataLoader()  # split officiel : adult.data = train, adult.test = test
X_train, X_test, y_train, y_test = loader.load_and_split()

print(f"Train : {X_train.shape[0]:,} obs  -  Test : {X_test.shape[0]:,} obs")
print(f"Features brutes : {X_train.shape[1]}")
X_train.head()


On ajuste **tout sur le train**. La cible textuelle (`>50K` / `<=50K`) est
encodée en 0/1 via le `LabelEncoder` interne de la pipeline.

In [ ]:
pipe = PreprocessingPipeline(scaling=True)

y_train_enc = pipe.fit_transform_target(y_train)
y_test_enc = pipe.transform_target(y_test)

print("Classes (ordre LabelEncoder) :", list(pipe.classes_))
print("Répartition train encodée   :", np.bincount(y_train_enc))


## 2. La pipeline de preprocessing — vue d'ensemble

`build()` retourne un `ColumnTransformer` prêt à être ajusté. On affiche son
diagramme : chaque branche cible un sous-ensemble de colonnes (les features
listées dans `FEATURES_TO_DROP` sont ignorées via `remainder="drop"`).

In [ ]:
print("Numériques normales (StandardScaler) :", NORMAL_NUMERICAL_FEATURES)
print("Numériques log1p (RobustScaler)       :", LOG_NUMERICAL_FEATURES)
print("Indicateurs binaires has_*            :", BINARY_NUMERICAL_FEATURES)
print("Catégorielles one-hot (mode)          :", MODE_CATEGORICAL_FEATURES)
print("Catégorielles one-hot + 'Unknown'     :", UNKNOWN_ONEHOT_FEATURES)
print(f"Catégorielles regroupées (<{RARE_CATEGORY_MIN_FREQUENCY})        :", GROUPED_CATEGORICAL_FEATURES)
print("Features ignorées (drop)              :", FEATURES_TO_DROP)

pre = pipe.build()
pre


## 3. Application de la pipeline (`fit_transform` sur le train)

In [ ]:
X_train_t = pre.fit_transform(X_train, y_train_enc)
feat_names = pre.get_feature_names_out()
X_train_t = pd.DataFrame(X_train_t, columns=feat_names, index=X_train.index)

print(f"Avant : {X_train.shape[1]} colonnes  ->  Apres : {X_train_t.shape[1]} colonnes")
X_train_t.head()


Le `OneHotEncoder` produit une colonne binaire par modalité (la dimensionnalité
augmente). Toutes les sorties sont désormais **numériques**, prêtes pour un estimateur.

## 4. Vérification des effets, décision par décision

Pour chaque décision du **bilan AED**, on contrôle que la pipeline l'applique correctement.
On privilégie les **tableaux** (constats vérifiables) ; on ne garde un graphe que lorsqu'il
montre un effet **non trivial** (le `log1p` sur `capital-gain`).

### 4.1 Valeurs manquantes & `?`

Les `?` du dataset Adult sont convertis en `NaN` **dès le chargement** (`DataLoader`).
La pipeline garantit qu'il ne reste **aucun manquant** en sortie, mais avec une
stratégie **par variable** plutôt qu'une imputation uniforme :

- **`workclass` / `occupation`** → catégorie **`Unknown`** (manquance informative, conservée) ;
- **`native-country`** → imputation par le **mode** (US, manquance non informative).


In [ ]:
# Les ? sont deja convertis en NaN au chargement -> on compte les NaN
raw_missing = X_train[CATEGORICAL_FEATURES].isna().sum()
print("Valeurs manquantes (NaN) dans le train brut :")
print(raw_missing[raw_missing > 0].to_string())

n_nan = int(np.isnan(X_train_t.values).sum())
print(f"\nNaN dans la matrice transformee : {n_nan}")


### 4.2 Mise à l'échelle des variables numériques

Le `StandardScaler` recentre/réduit les numériques « normales » (indispensable pour les
modèles sensibles à l'échelle : régression logistique, SVM, KNN). La **forme** des
distributions est déjà décrite dans l'EDA — ici on **vérifie** simplement qu'en sortie la
**moyenne ≈ 0** et l'**écart-type ≈ 1**.

In [ ]:
# Vérification du StandardScaler : moyenne ≈ 0, écart-type ≈ 1
scaled = X_train_t[[f"num__{c}" for c in NORMAL_NUMERICAL_FEATURES]].copy()
scaled.columns = NORMAL_NUMERICAL_FEATURES
scaled.agg(["mean", "std"]).round(3)

### 4.3 Variables `capital-*` : log1p + indicateurs binaires (proposition *hurdle*)

`capital-gain` et `capital-loss` sont **zero-inflated** (>90 % de zéros) et très
asymétriques. La proposition AED les décompose façon « hurdle » :

- **`capital-gain`** → `log1p` + `RobustScaler` : on comprime la queue lourde
  tout en **conservant la magnitude** (un gros gain est très prédictif de `>50K`).
- **`capital-loss`** → **pas** de log : sa magnitude est concentrée et peu
  informative, un simple indicateur binaire suffit.
- **Indicateurs `has_capital-gain` / `has_capital-loss`** (branche `num_bin`) :
  captent le signal **zéro / non-zéro**, qui porte l'essentiel de l'information.

> ⚠️ C'est la **proposition AED**. L'ablation (`modeling §3`) la compare au traitement
> **`naive`** et **retient le naïf** (le hurdle dégrade les arbres). Le graphe ci-dessous
> illustre donc la proposition, **pas** le choix final.

In [ ]:
# capital-gain : brut vs log1p + RobustScaler
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(X_train["capital-gain"], ax=axes[0], bins=40, color="steelblue")
axes[0].set_title("capital-gain - brut")
axes[0].set_yscale("log")
sns.histplot(X_train_t["num_log__capital-gain"], ax=axes[1], bins=40, color="seagreen")
axes[1].set_title("capital-gain - log1p + RobustScaler")
fig.suptitle("capital-gain : log1p conserve la magnitude", fontsize=14)
fig.tight_layout()
plt.show()

# Indicateurs binaires has_* (capital-gain & capital-loss)
bin_cols = [f"num_bin__has_{c}" for c in BINARY_NUMERICAL_FEATURES]
prop = X_train_t[bin_cols].mean().rename("proportion à 1").round(4)
print("Part d'individus avec un capital non nul :")
print(prop.to_string())


### 4.4 Encodage one-hot des catégorielles (dont `workclass`/`occupation` → `Unknown`)

Chaque variable catégorielle est éclatée en une colonne binaire par modalité.
Deux branches one-hot :

- **`cat`** (marital-status, relationship, race, sex) : imputation par le **mode**
  (sans effet ici, ces colonnes n'ont aucun manquant) puis one-hot. **Toutes** les
  modalités sont conservées.
- **`cat_unknown`** (`workclass`, `occupation`) : leur manquance est **informative**,
  donc les `?` deviennent une catégorie **`Unknown`** à part entière (`workclass_Unknown`,
  `occupation_Unknown`) au lieu d'être imputés par le mode. Le signal est conservé ;
  `occupation` (15 modalités) reste gérable en one-hot.

In [ ]:
# Branches one-hot : cat (mode) + cat_unknown (workclass/occupation -> Unknown)
onehot_bases = MODE_CATEGORICAL_FEATURES + UNKNOWN_ONEHOT_FEATURES
onehot_cols = [c for c in feat_names if c.startswith("cat__") or c.startswith("cat_unknown__")]
print(f"{len(onehot_bases)} variables one-hot  ->  {len(onehot_cols)} colonnes\n")
for base in onehot_bases:
    n = sum(f"__{base}_" in c for c in feat_names)
    print(f"  {base:<16} -> {n:>3} modalites")

# Présence de la catégorie dédiée Unknown pour workclass
unknown_col = "cat_unknown__workclass_Unknown"
print(f"\n{unknown_col} : {int(X_train_t[unknown_col].sum())} individus "
      f"(= manquants workclass conservés, pas imputés)")


### 4.5 `native-country` : regroupement des modalités rares

`native-country` a **41 modalités** (~90 % US), beaucoup de bruit sur les faibles
effectifs. Manquance **non informative** → imputation par le **mode** (US). Ses
modalités `<50` obs sont **regroupées** (`min_frequency = RARE_CATEGORY_MIN_FREQUENCY`)
dans un bac `infrequent_sklearn`, puis one-hot.

In [ ]:
# native-country : effet du regroupement des modalites rares (<50 obs)
country_cols = [c for c in feat_names if c.startswith("cat_country__")]
n_raw = X_train["native-country"].nunique(dropna=True)
print(f"native-country : {n_raw} modalites brutes  ->  {len(country_cols)} colonnes one-hot")
print("Bac des modalites rares present :", any("infrequent" in c for c in country_cols))

## 5. Transformation cohérente du jeu de test

Le test est transformé avec `transform` (et **non** `fit_transform`) : on
réutilise les statistiques apprises sur le train (médianes, moyennes/écarts,
modalités, encodage cible). Aucune fuite, et le test produit exactement les
mêmes colonnes.

In [ ]:
X_test_t = pre.transform(X_test)

print(f"Train transforme : {X_train_t.shape}")
print(f"Test  transforme : {X_test_t.shape}")
print("Memes colonnes           :", X_test_t.shape[1] == X_train_t.shape[1])
print("NaN dans le test         :", int(X_test_t.isna().sum().sum()))


## 6. Récapitulatif

- **`?` → NaN (au chargement) → traitement par variable** : plus aucun manquant en sortie.
- **Numériques normales** (age, education-num, hours-per-week) : recentrées/réduites (`StandardScaler`).
- **`capital-gain`** : `log1p` + `RobustScaler` (queue lourde conservée) **+** indicateur `has_capital-gain`.
- **`capital-loss`** : indicateur binaire `has_capital-loss` **seul** (pas de log).
- **Catégorielles faible cardinalité** (marital-status, relationship, race, sex) : one-hot, toutes modalités gardées.
- **`workclass` / `occupation`** : one-hot avec catégorie dédiée **`Unknown`** (manquance informative conservée).
- **`native-country`** : imputation mode (US) + regroupement des modalités `<50` obs, puis one-hot.
- **`education` / `fnlwgt`** : ignorées (`remainder="drop"`).
- La pipeline est **ajustée sur le train**, appliquée à l'identique au test → pas de data leakage.

> ⚠️ Ce récap décrit la pipeline **par défaut** (`hurdle`). Le **modèle final** utilise
> `capital_treatment="naive"` (cf. ablation `modeling §3`) : `capital-*` y sont de simples
> numériques mises à l'échelle, **sans `log1p` ni `has_*`**. Les autres traitements
> (suppressions, `Unknown`, regroupement pays, one-hot) sont, eux, conservés tels quels.